# Convert TSEncoder Model to CoreML

This notebook converts the trained PyTorch TSEncoder model to CoreML format for iOS deployment.

**Input:** `best_model.pt` (PyTorch checkpoint)  
**Output:** `TSEncoderPPG.mlpackage` (CoreML model for iOS)

## 1. Setup Environment and Imports

In [30]:
import sys
import torch
import numpy as np
import coremltools as ct
from pathlib import Path
import importlib

# Setup paths
project_root = Path.cwd().parent
ts2vec_path = project_root / 'ts2vec'

# Add to Python path
for path in [str(project_root), str(ts2vec_path)]:
    if path not in sys.path:
        sys.path.insert(0, path)

print("Environment Setup")
print("="*60)
print(f"PyTorch version: {torch.__version__}")
print(f"CoreMLTools version: {ct.__version__}")
print(f"Python version: {sys.version.split()[0]}")
print(f"\nProject root: {project_root}")
print(f"ts2vec exists: {ts2vec_path.exists()}")
print(f"encoder.py exists: {(ts2vec_path / 'models' / 'encoder.py').exists()}")
print("="*60)

Environment Setup
PyTorch version: 2.8.0
CoreMLTools version: 8.3.0
Python version: 3.10.18

Project root: /Users/omarhassan/Desktop/3x1-PPG
ts2vec exists: True
encoder.py exists: True


## 2. Import TSEncoder and Verify

In [31]:
# Import TSEncoder from ts2vec
try:
    from models.encoder import TSEncoder
    print("TSEncoder imported successfully")
    TSEncoder_available = True
except ImportError as e:
    print(f"TSEncoder import failed: {e}")
    TSEncoder_available = False
    raise

# Reload src.models to ensure it picks up TSEncoder
if TSEncoder_available:
    import src.models.ts2vec_ppg
    importlib.reload(src.models.ts2vec_ppg)
    from src.models.ts2vec_ppg import TS2VEC_AVAILABLE, build_tsencoder_ppg
    print(f"TS2VEC_AVAILABLE: {TS2VEC_AVAILABLE}")
    print(f"build_tsencoder_ppg imported")
    
    if not TS2VEC_AVAILABLE:
        raise ImportError("TS2VEC_AVAILABLE is False after import")

TSEncoder imported successfully
TS2VEC_AVAILABLE: True
build_tsencoder_ppg imported


## 3. Load Trained Model

In [32]:
# Build model architecture
print("Building model architecture...")
model = build_tsencoder_ppg(
    input_dims=1,
    output_dims=320,
    hidden_dims=64,
    depth=10,
    dropout=0.1
)

# Load trained weights
checkpoint_path = project_root / 'best_model.pt'
print(f"Loading checkpoint from: {checkpoint_path}")

checkpoint = torch.load(checkpoint_path, map_location='cpu')

# Handle different checkpoint formats
if isinstance(checkpoint, dict):
    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"Checkpoint info: Epoch {checkpoint.get('epoch', 'N/A')}, Val Loss: {checkpoint.get('val_loss', 'N/A'):.4f}")
    elif 'state_dict' in checkpoint:
        model.load_state_dict(checkpoint['state_dict'])
    else:
        model.load_state_dict(checkpoint)
else:
    model.load_state_dict(checkpoint)

model.eval()

print("\nModel loaded successfully")
params = model.count_parameters()
print(f"\nModel Parameters:")
print(f"  Total: {params['total']:,}")
print(f"  Encoder: {params['encoder']:,}")
print(f"  Regression head: {params['regression_head']:,}")

Building model architecture...
Loading checkpoint from: /Users/omarhassan/Desktop/3x1-PPG/best_model.pt
Checkpoint info: Epoch 4, Val Loss: 751.8808

Model loaded successfully

Model Parameters:
  Total: 701,569
  Encoder: 637,248
  Regression head: 64,321


## 4. Create and Test Example Input

In [33]:
# Create example input: single 1-second PPG segment (100 samples at 100Hz)
# Shape: (batch_size, sequence_length, features)
example_input = torch.randn(1, 100, 1)

print(f"Example input shape: {example_input.shape}")

# Test model with example input
with torch.no_grad():
    example_output = model(example_input)
    
print(f"Example output shape: {example_output.shape}")
print(f"Example output value: {example_output.item():.2f} mg/dL")
print("\nModel forward pass successful")

Example input shape: torch.Size([1, 100, 1])
Example output shape: torch.Size([1, 1])
Example output value: 129.09 mg/dL

Model forward pass successful


## 5. Convert to CoreML

In [34]:
# Create a TorchScript-compatible wrapper
# This wrapper avoids the problematic mask generation logic in TSEncoder

class CoreMLWrapper(torch.nn.Module):
    """Wrapper that fixes inference mode for CoreML conversion."""
    
    def __init__(self, original_model):
        super().__init__()
        self.encoder_input_fc = original_model.encoder.input_fc
        self.encoder_feature_extractor = original_model.encoder.feature_extractor
        self.encoder_repr_dropout = original_model.encoder.repr_dropout
        self.regression_head = original_model.regression_head
    
    def forward(self, x):
        # Input: B x T x input_dims
        if x.dim() == 2:
            x = x.unsqueeze(-1)
        
        # Replace NaN with 0
        x = torch.nan_to_num(x, nan=0.0)
        
        # Input projection
        x = self.encoder_input_fc(x)  # B x T x hidden_dims
        
        # No masking during inference - just process normally
        # Conv encoder expects B x Ch x T
        x = x.transpose(1, 2)  # B x hidden_dims x T
        x = self.encoder_repr_dropout(self.encoder_feature_extractor(x))  # B x output_dims x T
        x = x.transpose(1, 2)  # B x T x output_dims
        
        # Global average pooling
        x = torch.mean(x, dim=1)  # B x output_dims
        
        # Regression
        glucose = self.regression_head(x)  # B x 1
        return glucose

# Create wrapper
print("Creating CoreML-compatible wrapper...")
wrapper_model = CoreMLWrapper(model)
wrapper_model.eval()

# Test wrapper
with torch.no_grad():
    wrapper_output = wrapper_model(example_input)
    print(f"Wrapper output: {wrapper_output.item():.4f} mg/dL")
    print(f"Original output: {example_output.item():.4f} mg/dL")
    print(f"Match: {torch.allclose(wrapper_output, example_output, atol=1e-4)}")

# Convert to CoreML
input_shape = ct.Shape(shape=(1, 100, 1))

print("\nConverting to CoreML...")
print("This may take a minute...\n")

# Trace the wrapper model
traced_wrapper = torch.jit.trace(wrapper_model, example_input)

coreml_model = ct.convert(
    traced_wrapper,
    inputs=[
        ct.TensorType(
            name="ppg_signal",
            shape=input_shape,
            dtype=np.float32
        )
    ],
    convert_to="mlprogram",
    minimum_deployment_target=ct.target.iOS15
)

print("Conversion successful")

Creating CoreML-compatible wrapper...
Wrapper output: 129.0900 mg/dL
Original output: 129.0900 mg/dL
Match: True

Converting to CoreML...
This may take a minute...



Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 260.83 passes/s]


Conversion successful


## 6. Add Model Metadata

In [35]:
# Add metadata for better documentation
coreml_model.author = "Omar Hassan"
coreml_model.license = "MIT"
coreml_model.short_description = "TSEncoder model for blood glucose prediction from PPG signals"
coreml_model.version = "1.0.0"

# Add input/output descriptions
coreml_model.input_description["ppg_signal"] = "PPG signal segment (100 samples, 1 second at 100Hz)"

# Get actual output name from model
spec = coreml_model.get_spec()
output_name = spec.description.output[0].name
coreml_model.output_description[output_name] = "Predicted blood glucose level in mg/dL"

print("Metadata added successfully")
print(f"Output name: {output_name}")

Metadata added successfully
Output name: var_309


## 7. Test CoreML Model

In [36]:
# Test CoreML model with same input
print("Testing CoreML model...\n")

coreml_input = {"ppg_signal": example_input.numpy()}
coreml_output = coreml_model.predict(coreml_input)

pytorch_val = example_output.item()
# Get the output name dynamically
output_key = list(coreml_output.keys())[0]
coreml_val = coreml_output[output_key][0][0]

print(f"PyTorch output: {pytorch_val:.4f} mg/dL")
print(f"CoreML output:  {coreml_val:.4f} mg/dL")

# Check if outputs match (within tolerance)
diff = abs(coreml_val - pytorch_val)
print(f"\nDifference: {diff:.6f} mg/dL")

if diff < 0.01:
    print("CoreML model matches PyTorch model perfectly")
elif diff < 0.1:
    print("CoreML model matches PyTorch model within tolerance")
else:
    print("Warning: Outputs differ significantly")

Testing CoreML model...

PyTorch output: 129.0900 mg/dL
CoreML output:  127.8750 mg/dL

Difference: 1.214951 mg/dL


## 8. Test with Multiple Segments

In [37]:
# Test with multiple segments (as iOS app will do)
num_segments = 30
test_segments = torch.randn(num_segments, 100, 1)

print(f"Testing with {num_segments} segments...\n")

# PyTorch predictions
with torch.no_grad():
    pytorch_predictions = [model(segment.unsqueeze(0)).item() for segment in test_segments]

pytorch_mean = np.mean(pytorch_predictions)
pytorch_std = np.std(pytorch_predictions)

print(f"PyTorch predictions:")
print(f"  Mean: {pytorch_mean:.2f} mg/dL")
print(f"  Std:  {pytorch_std:.2f} mg/dL")

# CoreML predictions
coreml_predictions = []
for segment in test_segments:
    coreml_input = {"ppg_signal": segment.unsqueeze(0).numpy()}
    coreml_out = coreml_model.predict(coreml_input)
    coreml_predictions.append(coreml_out[output_key][0][0])

coreml_mean = np.mean(coreml_predictions)
coreml_std = np.std(coreml_predictions)

print(f"\nCoreML predictions:")
print(f"  Mean: {coreml_mean:.2f} mg/dL")
print(f"  Std:  {coreml_std:.2f} mg/dL")

mean_diff = abs(pytorch_mean - coreml_mean)
print(f"\nDifference in means: {mean_diff:.4f} mg/dL")

if mean_diff < 0.01:
    print("Excellent match across multiple predictions")

Testing with 30 segments...

PyTorch predictions:
  Mean: 128.16 mg/dL
  Std:  2.06 mg/dL

CoreML predictions:
  Mean: 126.99 mg/dL
  Std:  2.05 mg/dL

Difference in means: 1.1711 mg/dL


## 9. Save CoreML Model

In [38]:
# Save to project root
output_path = project_root / "TSEncoderPPG.mlpackage"
coreml_model.save(str(output_path))

print("CoreML model saved successfully")
print(f"\nLocation: {output_path}")

print("\n" + "="*60)
print("CONVERSION COMPLETE")
print("="*60)
print("\nNext steps:")
print("1. Copy TSEncoderPPG.mlpackage to iOS Xcode project")
print("2. Add to target membership in Xcode")
print("3. Xcode will auto-generate Swift interface")
print("4. Update ModelService.swift to use the model")
print("5. Test on iOS device")

CoreML model saved successfully

Location: /Users/omarhassan/Desktop/3x1-PPG/TSEncoderPPG.mlpackage

CONVERSION COMPLETE

Next steps:
1. Copy TSEncoderPPG.mlpackage to iOS Xcode project
2. Add to target membership in Xcode
3. Xcode will auto-generate Swift interface
4. Update ModelService.swift to use the model
5. Test on iOS device


## 10. Model Specification Summary

In [39]:
# Print model spec
spec = coreml_model.get_spec()

print("Model Specification")
print("="*60)
print(f"Author: {coreml_model.author}")
print(f"Version: {coreml_model.version}")
print(f"License: {coreml_model.license}")
print(f"Description: {coreml_model.short_description}")
print()
print(f"Model Type: ML Program (iOS 15+)")
print()
print(f"Input: ppg_signal")
print(f"  Shape: [1, 100, 1]")
print(f"  Description: {coreml_model.input_description['ppg_signal']}")
print()
print(f"Output: {output_key}")
print(f"  Description: Predicted blood glucose level in mg/dL")
print("="*60)

Model Specification
Author: Omar Hassan
Version: 1.0.0
License: MIT
Description: TSEncoder model for blood glucose prediction from PPG signals

Model Type: ML Program (iOS 15+)

Input: ppg_signal
  Shape: [1, 100, 1]
  Description: PPG signal segment (100 samples, 1 second at 100Hz)

Output: var_309
  Description: Predicted blood glucose level in mg/dL
